# Hadoop Ecosystem for Data Engineers
## Every Concept You Need to Know — From HDFS to the Modern Lakehouse

**What you'll learn:**
- The Big Data problem and why Hadoop was created
- HDFS: architecture, replication, block storage, read/write paths
- MapReduce: the programming model that started it all
- YARN: cluster resource management
- Hive: SQL on Hadoop (and why Spark replaced it for ETL)
- HBase: NoSQL wide-column store on HDFS
- ZooKeeper: distributed coordination
- Kafka: streaming platform (architecture, topics, partitions, consumer groups)
- Sqoop, Flume, Oozie: ingestion and orchestration
- Spark on YARN vs Standalone vs Kubernetes
- How Hadoop evolved into the modern Lakehouse
- Interview questions and answers

**Prerequisites:** Basic understanding of distributed systems concepts
**Estimated Time:** 6-8 hours

---

> Even in 2026, Hadoop knowledge is essential. Many companies still run Hadoop,
> interview questions reference it constantly, and understanding it makes you
> a better Spark/Databricks engineer because you understand WHY things work the way they do.

---

---
# Section 1: The Big Data Problem & Why Hadoop Exists

## Before Hadoop: The Scale Wall

In the early 2000s, companies hit a wall:
- Data grew faster than single-machine capacity
- Vertical scaling (bigger machines) became impossibly expensive
- A single disk reads at ~100 MB/s; reading 1 TB takes ~3 hours

## The Key Insight: Distribute BOTH Storage AND Compute

Instead of one massive machine, use THOUSANDS of commodity machines:
- Split data into chunks, spread across many machines (HDFS)
- Send computation TO the data instead of moving data to compute (MapReduce)
- This is the fundamental paradigm shift that enabled Big Data

## The 3 Vs of Big Data (What Hadoop Solves)

| V | Description | Example |
|---|-------------|---------|
| **Volume** | Terabytes to petabytes | Facebook: 600 TB/day in 2014 |
| **Velocity** | High-speed data arrival | Twitter: 500M tweets/day |
| **Variety** | Structured + unstructured | Logs, JSON, images, CSVs mixed |

## Hadoop's Core Philosophy

```
"Move computation to data, not data to computation"

Traditional: 1 TB data ---network---> compute node (SLOW)
Hadoop:      compute code ---network---> 1000 nodes with data chunks (FAST)
```

In [ ]:
# Conceptual demonstration: Why distributed processing is faster
import time

print("WHY DISTRIBUTED PROCESSING MATTERS")
print("=" * 60)

# Simulate processing 1 billion records
total_records = 1_000_000_000

# Single machine: processes 10M records/sec
single_machine_rate = 10_000_000  # records/sec
single_time = total_records / single_machine_rate

# 100-node cluster: each processes 10M records/sec on its portion
num_nodes = 100
records_per_node = total_records / num_nodes
cluster_time = records_per_node / single_machine_rate

print(f"  Processing {total_records:,} records:")
print(f"  Single machine (10M rec/s): {single_time:.0f} seconds ({single_time/60:.1f} minutes)")
print(f"  100-node cluster:           {cluster_time:.0f} seconds ({cluster_time/60:.1f} minutes)")
print(f"  Speedup: {single_time/cluster_time:.0f}x faster")
print()
print("  This is the power of horizontal scaling.")
print("  Hadoop made this possible on COMMODITY hardware (cheap servers).")
print()
print("  Modern equivalent:")
print("  - HDFS -> Delta Lake / Cloud Object Storage (S3, ADLS, GCS)")
print("  - MapReduce -> Apache Spark")
print("  - YARN -> Kubernetes / Databricks Serverless")

---
# Section 2: HDFS — Hadoop Distributed File System

## Architecture

HDFS splits files into **blocks** (default 128 MB) and distributes them across the cluster
with **replication** (default 3 copies) for fault tolerance.

```
CLIENT writes file (1 GB)
  |
  v
NAMENODE (master)                     <-- Metadata: which blocks on which nodes
  |  Decides: split into 8 blocks (128 MB each)
  |  Place block replicas across racks
  v
DATANODES (workers)
  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐
  │DataNode 1│  │DataNode 2│  │DataNode 3│  │DataNode 4│
  │Block A,C │  │Block A,B │  │Block B,D │  │Block C,D │
  │Block E,G │  │Block E,F │  │Block F,H │  │Block G,H │
  └──────────┘  └──────────┘  └──────────┘  └──────────┘

  Each block stored on 3 different DataNodes (replication factor = 3)
  If DataNode 2 dies, blocks A,B,E,F still exist on other nodes!
```

## Key HDFS Concepts

| Concept | Description | Default |
|---------|-------------|---------|
| **Block size** | Unit of storage | 128 MB (was 64 MB) |
| **Replication factor** | Copies of each block | 3 |
| **NameNode** | Master: tracks metadata (file -> blocks -> locations) | 1 (+ standby) |
| **DataNode** | Worker: stores actual data blocks | Many (100s-1000s) |
| **Rack awareness** | Replicas placed on different racks | Tolerates rack failure |
| **Write-once** | Files are append-only (no random writes) | By design |
| **Heartbeat** | DataNodes report health to NameNode | Every 3 seconds |

## HDFS Read Path

```
1. Client asks NameNode: "Where are blocks of /data/orders.csv?"
2. NameNode returns: Block 1 → DN1, DN3; Block 2 → DN2, DN4; ...
3. Client reads blocks DIRECTLY from DataNodes (no NameNode bottleneck!)
4. Client prefers DataNode on same rack (data locality)
```

## HDFS Write Path

```
1. Client asks NameNode to create file
2. NameNode allocates blocks, returns DataNode list for each
3. Client writes Block 1 to DN1
4. DN1 pipelines replica to DN2, DN2 pipelines to DN3
5. Acknowledgements flow back: DN3 -> DN2 -> DN1 -> Client
6. Client proceeds to next block
```

In [ ]:
# HDFS concepts in Python (simulated)
print("HDFS CONCEPTS -- Simulated")
print("=" * 60)

class SimulatedHDFS:
    """Demonstrates HDFS block splitting and replication logic."""

    def __init__(self, block_size_mb=128, replication_factor=3, num_datanodes=6):
        self.block_size_mb = block_size_mb
        self.replication = replication_factor
        self.num_datanodes = num_datanodes
        self.metadata = {}  # NameNode metadata

    def write_file(self, filename: str, size_mb: int):
        """Simulate writing a file to HDFS."""
        import math
        num_blocks = math.ceil(size_mb / self.block_size_mb)

        blocks = {}
        for i in range(num_blocks):
            block_name = f"{filename}_block_{i}"
            # Assign to 'replication' different DataNodes
            import random
            datanodes = random.sample(range(self.num_datanodes), self.replication)
            blocks[block_name] = datanodes

        self.metadata[filename] = {
            "size_mb": size_mb,
            "num_blocks": num_blocks,
            "blocks": blocks
        }
        return blocks

    def describe_file(self, filename: str):
        """Show how file is stored across the cluster."""
        info = self.metadata[filename]
        print(f"  File: {filename}")
        print(f"  Size: {info['size_mb']} MB")
        print(f"  Blocks: {info['num_blocks']} x {self.block_size_mb} MB")
        print(f"  Replication: {self.replication}x")
        print(f"  Total storage used: {info['num_blocks'] * self.block_size_mb * self.replication} MB")
        print(f"  Block placement:")
        for block, nodes in info['blocks'].items():
            print(f"    {block} -> DataNodes {nodes}")


# Demo
hdfs = SimulatedHDFS(block_size_mb=128, replication_factor=3, num_datanodes=6)
hdfs.write_file("/data/orders.parquet", size_mb=500)
hdfs.describe_file("/data/orders.parquet")

print(f"\n  Key insight: 500 MB file uses {500 * 3} MB actual storage (3x replication)")
print("  But any 2 DataNodes can fail and data is STILL available!")

In [ ]:
# HDFS Shell Commands (reference -- run on a Hadoop cluster)
print("HDFS SHELL COMMANDS (reference)")
print("=" * 60)

commands = [
    ("hdfs dfs -ls /data/", "List files in HDFS directory"),
    ("hdfs dfs -put local.csv /data/", "Upload local file to HDFS"),
    ("hdfs dfs -get /data/file.csv ./", "Download from HDFS to local"),
    ("hdfs dfs -cat /data/file.csv | head", "View file contents"),
    ("hdfs dfs -mkdir -p /data/raw/2024/", "Create directory"),
    ("hdfs dfs -rm -r /data/temp/", "Delete directory recursively"),
    ("hdfs dfs -du -s -h /data/", "Check directory size"),
    ("hdfs dfs -count /data/", "Count files and directories"),
    ("hdfs dfs -chmod 755 /data/shared/", "Change permissions"),
    ("hdfs dfs -setrep -w 2 /data/temp.csv", "Change replication factor"),
    ("hdfs fsck /data/ -files -blocks", "Check filesystem health"),
    ("hdfs dfsadmin -report", "Cluster status report"),
]

for cmd, desc in commands:
    print(f"  {cmd}")
    print(f"    -> {desc}")
    print()

print("  Modern equivalent: dbutils.fs.ls('/mnt/data/') in Databricks")
print("  Or: aws s3 ls s3://bucket/prefix/ for cloud storage")

---
# Section 3: MapReduce — The Programming Model

## How MapReduce Works

MapReduce processes data in two phases:

```
INPUT DATA (distributed across HDFS)
     |
     v
MAP PHASE: Each mapper processes one block independently
     |  Input: (key, value) pairs
     |  Output: (intermediate_key, intermediate_value) pairs
     v
SHUFFLE & SORT: Framework groups by intermediate key
     |  All values for same key go to same reducer
     v
REDUCE PHASE: Each reducer aggregates one group of keys
     |  Input: (key, [list of values])
     |  Output: (key, aggregated_value)
     v
OUTPUT (written to HDFS)
```

## Classic Example: Word Count

```
Input file: "hello world hello spark hello world"

MAP (each word → count 1):
  ("hello", 1), ("world", 1), ("hello", 1), ("spark", 1), ("hello", 1), ("world", 1)

SHUFFLE & SORT (group by key):
  "hello" → [1, 1, 1]
  "spark" → [1]
  "world" → [1, 1]

REDUCE (sum values):
  ("hello", 3), ("spark", 1), ("world", 2)
```

## Why MapReduce Was Revolutionary (and Why Spark Replaced It)

| MapReduce | Spark |
|-----------|-------|
| Writes to HDFS after EVERY stage | Keeps data in MEMORY between stages |
| One Map + One Reduce per job | Arbitrary DAG of transformations |
| Java-heavy, verbose code | Python/Scala/SQL APIs |
| Minutes for simple queries | Seconds for same queries |
| Good for batch ETL | Good for batch + interactive + streaming |

In [ ]:
# MapReduce in Python (conceptual implementation)
from collections import defaultdict
from typing import List, Tuple

print("MAPREDUCE: Word Count (Python Implementation)")
print("=" * 60)

# Simulated input "files" (in HDFS these would be blocks)
input_splits = [
    "hello world hello spark",
    "hello databricks world hadoop",
    "spark streaming real time data",
    "hadoop hdfs mapreduce yarn hive",
]

# MAP PHASE: Process each split independently
def mapper(text: str) -> List[Tuple[str, int]]:
    """Map function: emit (word, 1) for each word."""
    return [(word.lower(), 1) for word in text.split()]

print("MAP PHASE:")
all_pairs = []
for i, split in enumerate(input_splits):
    pairs = mapper(split)
    all_pairs.extend(pairs)
    print(f"  Mapper {i}: {split[:30]}... -> {len(pairs)} pairs")

# SHUFFLE & SORT: Group by key (done by framework)
def shuffle_sort(pairs: List[Tuple[str, int]]) -> dict:
    """Group all values by key."""
    groups = defaultdict(list)
    for key, value in pairs:
        groups[key].append(value)
    return dict(sorted(groups.items()))

print(f"\nSHUFFLE & SORT:")
grouped = shuffle_sort(all_pairs)
print(f"  {len(grouped)} unique keys grouped")
for key in list(grouped.keys())[:5]:
    print(f"    '{key}' -> {grouped[key]}")

# REDUCE PHASE: Aggregate each group
def reducer(key: str, values: List[int]) -> Tuple[str, int]:
    """Reduce function: sum all counts for a word."""
    return (key, sum(values))

print(f"\nREDUCE PHASE:")
results = [reducer(k, v) for k, v in grouped.items()]
results.sort(key=lambda x: x[1], reverse=True)
for word, count in results[:10]:
    print(f"  {word}: {count}")

print(f"\n  Total unique words: {len(results)}")
print(f"  Total word occurrences: {sum(c for _, c in results)}")
print("\n  In real Hadoop: each phase runs on different machines in parallel!")

In [ ]:
# MapReduce patterns that DE should know
print("\nMAPREDUCE PATTERNS (commonly asked in interviews)")
print("=" * 60)

# Pattern 1: Aggregation (GROUP BY equivalent)
print("1. AGGREGATION (like SQL GROUP BY):")
orders = [
    ("customer_A", 100), ("customer_B", 200), ("customer_A", 150),
    ("customer_C", 300), ("customer_B", 250), ("customer_A", 175),
]
# Map: identity (already key-value pairs)
# Reduce: sum by customer
from collections import defaultdict
totals = defaultdict(float)
for customer, amount in orders:
    totals[customer] += amount
for cust, total in sorted(totals.items()):
    print(f"  {cust}: ${total:.0f}")

# Pattern 2: Filter (WHERE equivalent)
print("\n2. FILTER (like SQL WHERE):")
print("  Map emits only records matching condition")
print("  Reduce is identity (no aggregation needed)")
filtered = [(c, a) for c, a in orders if a > 150]
print(f"  Orders > $150: {filtered}")

# Pattern 3: Join (JOIN equivalent)
print("\n3. JOIN (Reduce-Side Join):")
print("  Map: tag each record with source table")
print("  Shuffle: group by join key")
print("  Reduce: combine records from both tables")
print("  This is EXACTLY what Spark's Sort-Merge Join does!")

# Pattern 4: Secondary Sort
print("\n4. SECONDARY SORT:")
print("  Sort by key (for grouping) AND value (for ordering within group)")
print("  Used for: top-N per group, time-ordered events per user")

---
# Section 4: YARN — Yet Another Resource Negotiator

## Cluster Resource Management

YARN is the operating system of the Hadoop cluster. It manages:
- **Who** gets to run (application scheduling)
- **How much** resources they get (CPU, memory allocation)
- **Where** they run (node selection, locality)

```
YARN ARCHITECTURE:
┌─────────────────────────────────────────────────────┐
│                 RESOURCE MANAGER                      │
│  (Master: tracks cluster resources, schedules apps)  │
│  ┌──────────────────┐  ┌────────────────────────┐   │
│  │    Scheduler     │  │  Application Manager   │   │
│  │(FIFO/Fair/Capacity│  │ (launches App Masters) │   │
│  └──────────────────┘  └────────────────────────┘   │
└─────────────────────────────────────────────────────┘
         |                    |                    |
┌────────────────┐  ┌────────────────┐  ┌────────────────┐
│  NODE MANAGER  │  │  NODE MANAGER  │  │  NODE MANAGER  │
│   (Worker 1)   │  │   (Worker 2)   │  │   (Worker 3)   │
│ ┌────────────┐ │  │ ┌────────────┐ │  │ ┌────────────┐ │
│ │ Container  │ │  │ │ Container  │ │  │ │ Container  │ │
│ │ (App Task) │ │  │ │ (App Task) │ │  │ │ (App Task) │ │
│ └────────────┘ │  │ └────────────┘ │  │ └────────────┘ │
│ ┌────────────┐ │  │ ┌────────────┐ │  │                │
│ │ Container  │ │  │ │ App Master │ │  │                │
│ │ (App Task) │ │  │ │  (Spark)   │ │  │                │
│ └────────────┘ │  │ └────────────┘ │  │                │
└────────────────┘  └────────────────┘  └────────────────┘
```

## Key YARN Concepts

| Concept | Role | Analogy |
|---------|------|---------|
| **ResourceManager** | Cluster master, allocates resources | Airport control tower |
| **NodeManager** | Per-node agent, manages containers | Gate agent |
| **ApplicationMaster** | Per-app coordinator | Pilot for your flight |
| **Container** | Allocated CPU + memory on a node | Seat on the plane |
| **Queue** | Resource pool for a team/project | Terminal in the airport |

## YARN Schedulers

| Scheduler | Behavior | Use Case |
|-----------|----------|----------|
| **FIFO** | First come, first served | Development clusters |
| **Fair** | Equal share to all running apps | Multi-tenant clusters |
| **Capacity** | Guaranteed minimum per queue | Enterprise (team quotas) |

## Spark on YARN

```
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  --num-executors 10 \
  --executor-memory 8g \
  --executor-cores 4 \
  my_pipeline.py

# This tells YARN:
# "I need 10 containers, each with 8GB RAM and 4 cores"
# YARN finds nodes with capacity and launches executor JVMs
```

In [ ]:
# YARN concepts: resource allocation simulation
print("YARN RESOURCE ALLOCATION")
print("=" * 60)

class YARNSimulator:
    """Demonstrates YARN resource allocation logic."""

    def __init__(self, total_memory_gb: int, total_cores: int, num_nodes: int):
        self.total_memory = total_memory_gb
        self.total_cores = total_cores
        self.num_nodes = num_nodes
        self.memory_per_node = total_memory_gb // num_nodes
        self.cores_per_node = total_cores // num_nodes
        self.allocated = []

    def submit_app(self, app_name: str, executors: int, exec_memory_gb: int, exec_cores: int):
        """Simulate submitting an application to YARN."""
        total_mem_needed = executors * exec_memory_gb
        total_cores_needed = executors * exec_cores

        can_run = (total_mem_needed <= self.total_memory and
                   total_cores_needed <= self.total_cores)

        result = {
            "app": app_name,
            "executors": executors,
            "memory_per_exec": f"{exec_memory_gb}GB",
            "cores_per_exec": exec_cores,
            "total_memory": f"{total_mem_needed}GB / {self.total_memory}GB",
            "total_cores": f"{total_cores_needed} / {self.total_cores}",
            "status": "ACCEPTED" if can_run else "REJECTED (insufficient resources)"
        }
        if can_run:
            self.allocated.append(result)
            self.total_memory -= total_mem_needed
            self.total_cores -= total_cores_needed
        return result

# Simulate a 10-node cluster (each: 64GB RAM, 16 cores)
yarn = YARNSimulator(total_memory_gb=640, total_cores=160, num_nodes=10)
print(f"  Cluster: 10 nodes x 64GB x 16 cores = 640GB, 160 cores total")
print()

# Submit applications
apps = [
    ("spark_daily_etl", 20, 8, 4),      # 20 executors, 8GB each, 4 cores
    ("spark_ml_training", 10, 16, 4),    # 10 executors, 16GB each, 4 cores
    ("hive_query", 5, 4, 2),             # 5 executors, 4GB each, 2 cores
    ("spark_streaming", 30, 8, 4),       # Will this fit?
]

for name, execs, mem, cores in apps:
    result = yarn.submit_app(name, execs, mem, cores)
    print(f"  {result['app']:<20} {result['executors']} exec x {result['memory_per_exec']} x {result['cores_per_exec']} cores")
    print(f"    Resources: {result['total_memory']} memory, {result['total_cores']} cores")
    print(f"    Status: {result['status']}")
    print()

---
# Section 5: Hive — SQL on Hadoop

## What is Hive?

Hive provides a SQL interface to data stored in HDFS. It translates SQL
queries into MapReduce (or Tez/Spark) jobs.

```
User writes SQL:
  SELECT department, AVG(salary) FROM employees GROUP BY department

Hive compiles to:
  MapReduce Job 1: Scan employees table from HDFS
    Map: emit (department, salary)
    Reduce: calculate AVG per department
  Write results to HDFS
```

## Hive Architecture

| Component | Role |
|-----------|------|
| **Metastore** | Stores table schemas, partition info, locations |
| **Driver** | Parses SQL, creates execution plan |
| **Execution Engine** | Runs plan (MapReduce, Tez, or Spark) |
| **SerDe** | Serializer/Deserializer for file formats |

## Hive Metastore — Still Alive in 2026!

The Hive Metastore is arguably Hive's most lasting contribution.
Even Spark, Presto, Trino, and Databricks can use it as a catalog.

```
Hive Metastore stores:
- Database names
- Table names, columns, types
- Partition information
- File locations (HDFS paths, S3 paths)
- Statistics (row counts, file sizes)

This is what Unity Catalog REPLACED in Databricks.
```

## Hive vs Spark SQL vs Databricks SQL

| Feature | Hive | Spark SQL | Databricks SQL |
|---------|------|-----------|----------------|
| Engine | MapReduce/Tez | Spark (in-memory) | Photon (C++ vectorized) |
| Speed | Minutes | Seconds | Sub-second |
| Interactive | No | Somewhat | Yes |
| ACID | Limited (Hive 3+) | Via Delta Lake | Full (Delta native) |
| Use today | Legacy systems | ETL pipelines | Analytics + ETL |

## Key Hive Concepts for Interviews

### Partitioning
```sql
-- Physical directories on HDFS
CREATE TABLE orders (order_id INT, amount DECIMAL)
PARTITIONED BY (year INT, month INT);

-- HDFS layout:
-- /warehouse/orders/year=2023/month=01/data.parquet
-- /warehouse/orders/year=2023/month=02/data.parquet
-- Query with partition filter only reads relevant directories!
```

### Bucketing
```sql
-- Hash-distribute rows into fixed number of files
CREATE TABLE orders (order_id INT, customer_id INT, amount DECIMAL)
CLUSTERED BY (customer_id) INTO 32 BUCKETS;

-- Benefit: joins on customer_id only compare matching buckets
-- Modern equivalent: Liquid Clustering in Delta Lake
```

In [ ]:
# Hive concepts mapped to modern Databricks
print("HIVE CONCEPTS -> MODERN EQUIVALENTS")
print("=" * 60)

comparisons = [
    ("Hive Metastore", "Unity Catalog", "Table metadata and schema registry"),
    ("Hive Partitioning", "Delta Lake Partitioning", "Physical data organization by column"),
    ("Hive Bucketing", "Liquid Clustering (Delta)", "Hash-based data co-location"),
    ("ORC/Parquet (Hive)", "Delta Lake (Parquet+log)", "Columnar storage with ACID"),
    ("Hive ACID (v3)", "Delta Lake ACID", "Transactional table operations"),
    ("HiveQL", "Spark SQL / Databricks SQL", "SQL dialect for big data"),
    ("Hive UDFs (Java)", "PySpark UDFs / Pandas UDFs", "Custom functions"),
    ("Hive Views", "Spark SQL Views / DLT", "Virtual tables / materialized views"),
    ("Hive Statistics", "Delta Statistics / ANALYZE", "Optimizer metadata"),
    ("Tez Execution", "Spark DAG + Photon", "Execution engine"),
]

print(f"  {'Hive (Legacy)':<25} {'Modern (Databricks)':<30} {'Purpose'}")
print(f"  {'-'*85}")
for hive, modern, purpose in comparisons:
    print(f"  {hive:<25} {modern:<30} {purpose}")

print()
print("  Key interview point: Hive Metastore is STILL used as the catalog")
print("  backend for many Spark clusters. Unity Catalog is Databricks' replacement.")

---
# Section 6: Apache Kafka — Distributed Streaming Platform

## Why Every DE Must Know Kafka

Kafka is THE standard for:
- Real-time data pipelines (CDC, event streaming)
- Decoupling producers and consumers
- Buffering between batch and streaming systems
- Event sourcing architectures

## Kafka Architecture

```
PRODUCERS                    KAFKA CLUSTER                 CONSUMERS
(write data)                 (stores & distributes)        (read data)

┌──────────┐   ┌────────────────────────────────────┐   ┌──────────┐
│ Web App  │──>│  BROKER 1    BROKER 2    BROKER 3  │──>│ Spark    │
│ (orders) │   │  ┌───────┐  ┌───────┐  ┌───────┐  │   │ Streaming│
└──────────┘   │  │Topic A│  │Topic A│  │Topic A│  │   └──────────┘
               │  │Part 0 │  │Part 1 │  │Part 2 │  │
┌──────────┐   │  └───────┘  └───────┘  └───────┘  │   ┌──────────┐
│ IoT      │──>│  ┌───────┐  ┌───────┐  ┌───────┐  │──>│ Flink    │
│ Sensors  │   │  │Topic B│  │Topic B│  │Topic B│  │   │ (alerts) │
└──────────┘   │  │Part 0 │  │Part 1 │  │Part 2 │  │   └──────────┘
               │  └───────┘  └───────┘  └───────┘  │
┌──────────┐   │                                    │   ┌──────────┐
│ CDC      │──>│  ZooKeeper / KRaft (metadata)      │──>│ Delta    │
│ (Debezium)│  │                                    │   │ (storage)│
└──────────┘   └────────────────────────────────────┘   └──────────┘
```

## Key Kafka Concepts

| Concept | Description |
|---------|-------------|
| **Topic** | Named stream of records (like a table) |
| **Partition** | Ordered, immutable sequence within a topic (parallelism unit) |
| **Offset** | Position of a message within a partition (like a row ID) |
| **Producer** | Writes messages to topics |
| **Consumer** | Reads messages from topics |
| **Consumer Group** | Set of consumers that divide partition reading (load balancing) |
| **Broker** | Kafka server that stores partitions |
| **Replication** | Each partition has leader + followers (fault tolerance) |
| **Retention** | How long messages are kept (default: 7 days) |

## Kafka Guarantees

| Guarantee | Meaning | Config |
|-----------|---------|--------|
| **At-most-once** | May lose messages, never duplicates | acks=0 |
| **At-least-once** | No loss, but possible duplicates | acks=all + retry |
| **Exactly-once** | No loss, no duplicates | Idempotent producer + transactions |

In [ ]:
# Kafka concepts simulation
from collections import defaultdict
import time

print("KAFKA CONCEPTS -- Simulated")
print("=" * 60)

class KafkaTopic:
    """Simulates a Kafka topic with partitions."""

    def __init__(self, name: str, num_partitions: int = 3):
        self.name = name
        self.partitions = {i: [] for i in range(num_partitions)}
        self.offsets = defaultdict(lambda: defaultdict(int))  # consumer_group -> partition -> offset

    def produce(self, key: str, value: dict):
        """Write a message (key determines partition)."""
        partition = hash(key) % len(self.partitions)
        offset = len(self.partitions[partition])
        self.partitions[partition].append({"key": key, "value": value, "offset": offset})
        return partition, offset

    def consume(self, consumer_group: str, partition: int, max_messages: int = 10):
        """Read messages from a partition starting at committed offset."""
        start_offset = self.offsets[consumer_group][partition]
        messages = self.partitions[partition][start_offset:start_offset + max_messages]
        self.offsets[consumer_group][partition] = start_offset + len(messages)
        return messages

    def describe(self):
        print(f"  Topic: {self.name}")
        print(f"  Partitions: {len(self.partitions)}")
        for p, msgs in self.partitions.items():
            print(f"    Partition {p}: {len(msgs)} messages")


# Simulate a data pipeline
topic = KafkaTopic("orders_events", num_partitions=3)

# Produce events (partitioned by customer_id)
events = [
    ("cust_1", {"event": "order_placed", "amount": 99.99}),
    ("cust_2", {"event": "order_placed", "amount": 250.00}),
    ("cust_1", {"event": "payment_received", "amount": 99.99}),
    ("cust_3", {"event": "order_placed", "amount": 75.00}),
    ("cust_2", {"event": "order_shipped", "amount": 250.00}),
    ("cust_1", {"event": "order_delivered", "amount": 99.99}),
]

print("Producing events:")
for key, value in events:
    partition, offset = topic.produce(key, value)
    print(f"  {key} -> Partition {partition}, Offset {offset}: {value['event']}")

print()
topic.describe()

# Consume (like a Spark Streaming job would)
print("\nConsuming (consumer group 'spark_streaming'):")
for p in range(3):
    messages = topic.consume("spark_streaming", p)
    if messages:
        print(f"  Partition {p}: read {len(messages)} messages")
        for msg in messages:
            print(f"    offset={msg['offset']}: {msg['key']} - {msg['value']['event']}")

print("\n  Key insight: Same customer always goes to same partition (ordering guaranteed)")
print("  Multiple consumers in a group each read different partitions (parallelism)")

---
# Section 7: The Full Hadoop Ecosystem

## Components Every DE Should Know

```
DATA INGESTION          STORAGE              PROCESSING           QUERY / SERVING
┌──────────┐         ┌──────────┐         ┌──────────┐         ┌──────────┐
│  Kafka   │         │   HDFS   │         │  Spark   │         │  Hive    │
│  Flume   │────────>│  (blocks,│────────>│  (batch +│────────>│  Presto  │
│  Sqoop   │         │   replicated)│     │   stream)│         │  Trino   │
│  NiFi    │         │          │         │          │         │  Impala  │
└──────────┘         └──────────┘         └──────────┘         └──────────┘
                          |                     |
                     ┌──────────┐         ┌──────────┐
                     │  HBase   │         │  YARN    │
                     │  (NoSQL) │         │(resources)│
                     └──────────┘         └──────────┘
                          |                     |
                     ┌──────────┐         ┌──────────┐
                     │ZooKeeper │         │  Oozie   │
                     │(coordinate)│       │(orchestrate)│
                     └──────────┘         └──────────┘
```

## Quick Reference: What Each Component Does

| Component | What It Does | Modern Replacement |
|-----------|-------------|-------------------|
| **HDFS** | Distributed file storage | S3, ADLS, GCS, Delta Lake |
| **MapReduce** | Batch processing framework | Apache Spark |
| **YARN** | Resource management | Kubernetes, Databricks Serverless |
| **Hive** | SQL on Hadoop, metastore | Spark SQL, Databricks SQL, Unity Catalog |
| **HBase** | NoSQL wide-column store | DynamoDB, Cassandra, Cosmos DB |
| **Kafka** | Streaming platform | Still Kafka! (Confluent, MSK, Event Hubs) |
| **ZooKeeper** | Distributed coordination | KRaft (Kafka), etcd (K8s) |
| **Sqoop** | RDBMS ↔ HDFS bulk transfer | Spark JDBC, Fivetran, Airbyte |
| **Flume** | Log collection/ingestion | Fluentd, Vector, Datadog |
| **Oozie** | Workflow orchestration | Airflow, Dagster, Databricks Workflows |
| **Pig** | Data transformation scripting | PySpark, dbt |
| **Impala** | Low-latency SQL | Databricks SQL, Snowflake |
| **Presto/Trino** | Federated SQL queries | Trino, Starburst, Databricks |

---
# Section 8: From Hadoop to the Modern Lakehouse

## The Evolution

```
ERA 1 (2006-2012): Hadoop
  - HDFS + MapReduce + Hive
  - Batch only, very slow
  - "Store everything, figure it out later"

ERA 2 (2014-2018): Spark + Data Lake
  - Spark replaces MapReduce (100x faster)
  - Cloud storage replaces HDFS (S3, ADLS)
  - Still no ACID, no schema enforcement
  - "Data swamp" problem (ungoverned, unreliable)

ERA 3 (2019-2022): Lakehouse
  - Delta Lake adds ACID to cloud storage
  - Unity Catalog adds governance
  - Combine best of data lake (cheap, schema-on-read)
    with best of data warehouse (ACID, performance, governance)

ERA 4 (2023-2026): AI Lakehouse
  - Photon engine (C++ vectorized)
  - Serverless compute
  - AI/ML integrated (Feature Store, Model Serving)
  - Liquid Clustering replaces partitioning
  - Predictive Optimization (auto-maintain tables)
```

## What Survived from Hadoop

| Concept | Hadoop Era | Lakehouse Era |
|---------|-----------|---------------|
| Distributed storage | HDFS | Cloud object storage + Delta Lake |
| Columnar format | ORC | Parquet (via Delta) |
| SQL interface | Hive | Spark SQL / Databricks SQL |
| Metadata catalog | Hive Metastore | Unity Catalog |
| Resource management | YARN | Kubernetes / Serverless |
| Streaming | Kafka | Still Kafka + Structured Streaming |
| Batch processing | MapReduce | Spark |
| Orchestration | Oozie | Airflow / Databricks Workflows |

## Key Takeaway for Interviews

> "Hadoop pioneered distributed data processing. The CONCEPTS remain the same --
> data partitioning, fault tolerance, data locality, schema-on-read, distributed joins.
> What changed is the IMPLEMENTATION: in-memory processing (Spark), cloud-native storage,
> ACID transactions (Delta Lake), and unified governance (Unity Catalog)."


---
# Section 9: Interview Questions & Answers

In [ ]:
# Common Hadoop interview questions for DE roles
print("HADOOP INTERVIEW QUESTIONS")
print("=" * 60)

questions = [
    {
        "q": "What happens when a DataNode fails in HDFS?",
        "a": """The NameNode detects the failure via missed heartbeats (3s interval, 10min timeout).
It then re-replicates the blocks that were on the failed node to other DataNodes
to maintain the replication factor. This is automatic -- no manual intervention."""
    },
    {
        "q": "Why is HDFS block size 128MB (not smaller like 4KB)?",
        "a": """To minimize seek time relative to transfer time. With 128MB blocks:
- Seek time (~10ms) is negligible compared to transfer time (~1s at 100MB/s)
- Fewer blocks = less NameNode memory for metadata
- Larger blocks = more efficient sequential reads
Tradeoff: small files waste space (a 1KB file still uses one block entry)."""
    },
    {
        "q": "Explain the difference between MapReduce and Spark.",
        "a": """MapReduce writes intermediate results to HDFS disk after every stage.
Spark keeps data in-memory between stages (RDDs/DataFrames).
This makes Spark 10-100x faster for iterative algorithms (ML, multi-stage ETL).
Spark also supports a richer DAG of operations (not just map+reduce)."""
    },
    {
        "q": "What is data locality and why does it matter?",
        "a": """Data locality = running computation on the same node where the data is stored.
Levels: PROCESS_LOCAL > NODE_LOCAL > RACK_LOCAL > ANY.
Avoids network transfer of data (the most expensive operation in distributed systems).
Spark scheduler tries to assign tasks to nodes that have the data partition."""
    },
    {
        "q": "How does Kafka guarantee message ordering?",
        "a": """Ordering is guaranteed WITHIN a partition (not across partitions).
Same key always goes to same partition (via hash(key) % num_partitions).
So all events for customer_123 are ordered, but events across customers are not.
If you need global ordering: use a single partition (sacrifices parallelism)."""
    },
    {
        "q": "What is the NameNode single point of failure problem?",
        "a": """If the NameNode dies, the ENTIRE cluster is inaccessible (metadata is lost).
Solutions: NameNode HA (High Availability) with active/standby pair,
shared edit log (via JournalNodes or NFS), automatic failover via ZooKeeper.
Modern: Cloud storage (S3) eliminates this problem entirely."""
    },
    {
        "q": "Why did the industry move from Hadoop to cloud data platforms?",
        "a": """1. Separation of storage and compute (scale independently)
2. No cluster management overhead (serverless)
3. Pay-per-use vs always-on clusters
4. ACID transactions (Delta Lake vs raw HDFS)
5. Better governance (Unity Catalog vs Hive Metastore)
6. Much faster (Photon, in-memory vs disk-based MapReduce)"""
    },
]

for i, qa in enumerate(questions, 1):
    print(f"\nQ{i}: {qa['q']}")
    print(f"{'─' * 60}")
    for line in qa['a'].strip().split('\n'):
        print(f"  {line}")

---
# Summary

## Hadoop Ecosystem Cheat Sheet

| Component | One-Line Description | Still Relevant? |
|-----------|---------------------|-----------------|
| HDFS | Distributed file system (blocks + replication) | Concepts yes, tech replaced by cloud storage |
| MapReduce | Map-Shuffle-Reduce batch processing | Concepts yes, replaced by Spark |
| YARN | Cluster resource manager | Replaced by K8s/Serverless |
| Hive | SQL on Hadoop + Metastore | Metastore still used, SQL replaced by Spark SQL |
| HBase | NoSQL on HDFS | Niche, mostly replaced by cloud NoSQL |
| Kafka | Distributed streaming (topics, partitions) | VERY relevant, still the standard |
| ZooKeeper | Distributed coordination | Being replaced by KRaft in Kafka |
| Spark | In-memory distributed processing | THE standard for big data processing |

## What to Say in Interviews

> "I understand the Hadoop ecosystem and its evolution. While I work primarily
> with Spark and Delta Lake on cloud platforms, I know that HDFS concepts
> (block storage, replication, data locality) underpin how Spark reads data,
> that MapReduce patterns (map, shuffle, reduce) are exactly what happens
> during a Spark groupBy, and that Kafka remains essential for real-time pipelines.
> The shift to lakehouse architecture solved Hadoop's governance and performance
> limitations while keeping its core distributed processing principles."

---
*Hadoop Ecosystem for Data Engineers*